In [ ]:
# ── PHASE 1: Environment Check ───────────────────────────────────────
import yfinance, pandas, numpy, matplotlib, scipy, seaborn, sklearn
print("✓ All packages ready!")
print(f"  pandas {pandas.__version__} | numpy {numpy.__version__}")

In [ ]:
# ── PHASE 2: Data Ingestion ──────────────────────────────────────────
# Downloads 20 years of price data for 6 assets, saves to data/

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# ── Create output directories if they don't exist ────────────────────
# notebook lives in notebooks/ so we go one level up
os.makedirs('../data',    exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

# ── Assets ───────────────────────────────────────────────────────────
# SPY = S&P 500  |  EFA = Intl stocks  |  TLT = Long bonds
# IEF = Mid bonds  |  GLD = Gold  |  SHY = Short bonds (≈ cash)
ASSETS = ['SPY', 'EFA', 'TLT', 'IEF', 'GLD', 'SHY']
START  = '2005-01-01'
END    = '2024-12-31'

# ── Download; handle multi-level columns from newer yfinance ─────────
print("Downloading 20 years of price data...")
raw = yf.download(ASSETS, start=START, end=END, auto_adjust=True, progress=False)
prices = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw
prices = prices.dropna(axis=1, how='all').dropna()

print(f"✓ Assets: {list(prices.columns)}")
print(f"✓ {len(prices)} trading days  |  {prices.index[0].date()} → {prices.index[-1].date()}")

returns = prices.pct_change().dropna()
print(f"✓ Returns shape: {returns.shape}")

prices.to_csv('../data/prices.csv')
returns.to_csv('../data/returns.csv')
print("✓ Saved → data/prices.csv  &  data/returns.csv")

# ── Plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

(prices / prices.iloc[0] * 100).plot(ax=axes[0], linewidth=1)
axes[0].set_title('Normalised Asset Prices (base = 100)', fontsize=13)
axes[0].set_ylabel('Index')
axes[0].legend(ncol=3)
axes[0].grid(alpha=0.3)

returns['SPY'].plot(ax=axes[1], color='steelblue', linewidth=0.5, alpha=0.7)
axes[1].set_title('SPY Daily Returns', fontsize=13)
axes[1].set_ylabel('Daily return')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].grid(alpha=0.3)

spy_vol = returns['SPY'].rolling(30).std() * np.sqrt(252)
spy_vol.plot(ax=axes[2], color='crimson', linewidth=1)
axes[2].set_title('SPY Rolling 30-day Annualised Volatility', fontsize=13)
axes[2].set_ylabel('Volatility')
axes[2].axhline(spy_vol.mean(), color='black', linestyle='--', linewidth=0.8, label='Average')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/01_prices_and_vol.png', dpi=150)
plt.show()
print("\n✓ Phase 2 complete!  Saved → outputs/01_prices_and_vol.png")

In [ ]:
# ── PHASE 3: GMM Regime Classifier ──────────────────────────────────
# Gaussian Mixture Model detects 3 hidden market states:
#   Bull (low vol)  |  Bear (mid vol)  |  Crisis (high vol)

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

prices  = pd.read_csv('../data/prices.csv',  index_col=0, parse_dates=True)
returns = pd.read_csv('../data/returns.csv', index_col=0, parse_dates=True)

# ── Feature matrix ───────────────────────────────────────────────────
spy_ret = returns['SPY']
tlt_ret = returns['TLT']
vol_20  = spy_ret.rolling(20).std()  * np.sqrt(252)
mean_20 = spy_ret.rolling(20).mean() * 252

features = pd.DataFrame({
    'spy_return': spy_ret,
    'volatility': vol_20,
    'trend'     : mean_20,
    'tlt_return': tlt_ret,
}).dropna()

# Align to features index to prevent index-mismatch errors later
spy_ret_aligned = spy_ret.reindex(features.index)
vol_20_aligned  = vol_20.reindex(features.index)

print(f"✓ Feature matrix: {features.shape}  |  {list(features.columns)}")

# ── Fit GMM ──────────────────────────────────────────────────────────
scaler = StandardScaler()
X      = scaler.fit_transform(features)
gmm    = GaussianMixture(n_components=3, covariance_type='full',
                         n_init=20, random_state=42)
gmm.fit(X)
regime_labels = gmm.predict(X)
print(f"✓ GMM fitted!  Log-likelihood: {gmm.score(X):.4f}")

# ── Label by volatility rank ─────────────────────────────────────────
means     = scaler.inverse_transform(gmm.means_)
vol_idx   = list(features.columns).index('volatility')
order     = np.argsort(means[:, vol_idx])
label_map  = {order[0]: 'Bull', order[1]: 'Bear', order[2]: 'Crisis'}
colors_map = {'Bull': '#2ecc71', 'Bear': '#e67e22', 'Crisis': '#e74c3c'}
regimes    = pd.Series(regime_labels, index=features.index).map(label_map)

# ── Regime statistics ────────────────────────────────────────────────
print("\n── Regime Statistics ───────────────────────────────────────")
for name in ['Bull', 'Bear', 'Crisis']:
    mask    = regimes == name
    n_days  = mask.sum()
    avg_ret = spy_ret_aligned[mask].mean() * 252 * 100
    avg_vol = vol_20_aligned[mask].mean()  * 100
    print(f"  {name:8s}: {n_days:4d} days | "
          f"Avg annual return: {avg_ret:+.1f}% | "
          f"Avg vol: {avg_vol:.1f}%")

# ── Plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

spy_price = prices['SPY'].reindex(features.index)
ax = axes[0]
for date, regime in regimes.items():
    ax.axvspan(date, date + pd.Timedelta(days=1),
               color=colors_map[regime], alpha=0.3, linewidth=0)
spy_price.plot(ax=ax, color='black', linewidth=0.8)
ax.set_title('SPY Price with Detected Regimes', fontsize=13)
ax.set_ylabel('Price ($)')
ax.grid(alpha=0.2)
patches = [mpatches.Patch(color=colors_map[r], label=r) for r in ['Bull','Bear','Crisis']]
ax.legend(handles=patches, loc='upper left')

regime_num = regimes.map({'Bull': 1, 'Bear': 2, 'Crisis': 3})
ax = axes[1]
for regime, num, color in [('Bull',1,'#2ecc71'),('Bear',2,'#e67e22'),('Crisis',3,'#e74c3c')]:
    mask = regime_num == num
    ax.fill_between(regime_num.index, 0, mask.astype(int),
                    color=color, alpha=0.7, label=regime)
ax.set_title('Regime Timeline', fontsize=13)
ax.set_yticks([])
ax.legend(loc='upper left')
ax.grid(alpha=0.2)

ax = axes[2]
vol_20_aligned.plot(ax=ax, color='gray', linewidth=0.8, label='Volatility')
for regime, color in colors_map.items():
    mask = regimes == regime
    vol_20_aligned[mask].plot(ax=ax, color=color, linewidth=1.2, label=regime)
ax.set_title('Volatility coloured by Regime', fontsize=13)
ax.set_ylabel('Annualised Vol')
ax.legend(loc='upper right')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../outputs/02_regimes.png', dpi=150)
plt.show()
print("\n✓ Phase 3 complete!  Saved → outputs/02_regimes.png")

In [ ]:
# ── PHASE 4: Dynamic Portfolio Optimizer ────────────────────────────
# Finds optimal asset weights per regime via scipy.optimize.
# Objective: Bull → max Sharpe  |  Bear → balanced  |  Crisis → min vol

from scipy.optimize import minimize
import numpy as np
import matplotlib.pyplot as plt

def portfolio_performance(weights, mean_returns, cov_matrix):
    ret    = np.dot(weights, mean_returns) * 252
    vol    = np.sqrt(weights @ cov_matrix @ weights) * np.sqrt(252)
    sharpe = ret / vol if vol > 0 else 0
    return ret, vol, sharpe

def optimize_portfolio(mean_returns, cov_matrix, objective='sharpe'):
    n           = len(mean_returns)
    bounds      = [(0.05, 0.40)] * n
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    init_w      = np.ones(n) / n

    if objective == 'sharpe':
        fn = lambda w: -portfolio_performance(w, mean_returns, cov_matrix)[2]
    elif objective == 'min_vol':
        fn = lambda w:  portfolio_performance(w, mean_returns, cov_matrix)[1]
    elif objective == 'balanced':
        fn = lambda w: -(portfolio_performance(w, mean_returns, cov_matrix)[2]
                         - 0.5 * portfolio_performance(w, mean_returns, cov_matrix)[1])

    return minimize(fn, init_w, method='SLSQP',
                    bounds=bounds, constraints=constraints).x

ASSETS = list(returns.columns)
regime_objectives = {'Bull': 'sharpe', 'Bear': 'balanced', 'Crisis': 'min_vol'}
regime_weights    = {}

print("── Optimal Weights per Regime ──────────────────────────────")
for regime, objective in regime_objectives.items():
    mask        = regimes == regime
    regime_rets = returns.reindex(features.index)[mask]
    mean_r      = regime_rets.mean()
    cov_m       = regime_rets.cov()
    weights     = optimize_portfolio(mean_r.values, cov_m.values, objective)
    regime_weights[regime] = dict(zip(ASSETS, weights))

    ret, vol, sharpe = portfolio_performance(weights, mean_r.values, cov_m.values)
    print(f"\n  {regime} (objective: {objective})")
    for asset, w in zip(ASSETS, weights):
        print(f"    {asset:4s}  {w:.1%}  {'█' * int(w * 40)}")
    print(f"    → Expected return: {ret*100:.1f}%  |  Vol: {vol*100:.1f}%  |  Sharpe: {sharpe:.2f}")

# ── Bar chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
bar_colors = ['#3498db','#2ecc71','#e74c3c','#9b59b6','#f39c12','#1abc9c']

for ax, (regime, weights_dict) in zip(axes, regime_weights.items()):
    vals = list(weights_dict.values())
    bars = ax.bar(ASSETS, vals, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.set_title(f'{regime} Regime', fontsize=13, fontweight='bold',
                 color={'Bull':'#27ae60','Bear':'#e67e22','Crisis':'#e74c3c'}[regime])
    ax.set_ylabel('Weight')
    ax.set_ylim(0, 0.45)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(1/len(ASSETS), color='black', linestyle='--', linewidth=0.8, label='Equal weight')
    ax.legend(fontsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
                f'{val:.0%}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Dynamic Asset Allocation by Regime', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/03_regime_weights.png', dpi=150)
plt.show()
print("\n✓ Phase 4 complete!  Saved → outputs/03_regime_weights.png")

In [ ]:
# ── PHASE 5: Walk-Forward Backtester ────────────────────────────────
# Zero look-ahead bias: GMM trained only on past data at every step.
# 7 bps transaction cost applied on each monthly rebalance.

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TRANSACTION_COST = 0.0007
TRAIN_YEARS      = 3
REBAL_FREQ       = 21

prices  = pd.read_csv('../data/prices.csv',  index_col=0, parse_dates=True)
returns = pd.read_csv('../data/returns.csv', index_col=0, parse_dates=True)
ASSETS  = list(returns.columns)
dates   = returns.index
TRAIN_DAYS = TRAIN_YEARS * 252

print(f"Total days: {len(dates)}  |  Training window: {TRAIN_DAYS} days")
print(f"Backtest starts: {dates[TRAIN_DAYS].date()}")

portfolio_values = [1.0]
benchmark_values = [1.0]
portfolio_dates  = [dates[TRAIN_DAYS]]
regime_history   = []
prev_weights     = None

print("\nRunning walk-forward backtest...")

for i in range(TRAIN_DAYS, len(dates) - 1):
    train_ret = returns.iloc[i - TRAIN_DAYS:i]
    spy_r     = train_ret['SPY']
    tlt_r     = train_ret['TLT']
    vol_20    = spy_r.rolling(20).std()  * np.sqrt(252)
    mean_20   = spy_r.rolling(20).mean() * 252

    feats = pd.DataFrame({'spy_return': spy_r, 'volatility': vol_20,
                          'trend': mean_20, 'tlt_return': tlt_r}).dropna()

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(feats)
    gmm     = GaussianMixture(n_components=3, covariance_type='full',
                               n_init=5, random_state=42)
    gmm.fit(X_train)

    X_today   = scaler.transform(feats.iloc[[-1]])
    raw_label = gmm.predict(X_today)[0]
    means_inv = scaler.inverse_transform(gmm.means_)
    order     = np.argsort(means_inv[:, list(feats.columns).index('volatility')])
    regime    = {order[0]: 'Bull', order[1]: 'Bear', order[2]: 'Crisis'}[raw_label]
    regime_history.append(regime)

    weights = np.array([regime_weights[regime][a] for a in ASSETS])

    cost = 0.0
    if prev_weights is not None and (i - TRAIN_DAYS) % REBAL_FREQ == 0:
        cost = np.sum(np.abs(weights - prev_weights)) * TRANSACTION_COST
    prev_weights = weights.copy()

    next_ret = returns.iloc[i + 1].values
    portfolio_values.append(portfolio_values[-1] * (1 + np.dot(weights, next_ret) - cost))
    portfolio_dates.append(dates[i + 1])

    bm_w = np.array([0.6 if a=='SPY' else 0.4 if a=='TLT' else 0.0 for a in ASSETS])
    benchmark_values.append(benchmark_values[-1] * (1 + np.dot(bm_w, next_ret)))

    if (i - TRAIN_DAYS) % 252 == 0:
        print(f"  {dates[i].date()}  regime={regime:6s}  "
              f"portfolio=${portfolio_values[-1]:.2f}  "
              f"benchmark=${benchmark_values[-1]:.2f}")

print("\n✓ Backtest complete!")

results = pd.DataFrame({
    'portfolio': portfolio_values,
    'benchmark': benchmark_values,
    'regime'   : [''] + regime_history
}, index=portfolio_dates)
results.to_csv('../outputs/backtest_results.csv')

def compute_metrics(series, label):
    dr      = series.pct_change().dropna()
    ann_ret = ((series.iloc[-1]/series.iloc[0])**(252/len(dr))-1)*100
    ann_vol = dr.std()*np.sqrt(252)*100
    sharpe  = ann_ret/ann_vol
    dd      = (series - series.cummax())/series.cummax()
    max_dd  = dd.min()*100
    sortino_d = dr[dr<0].std()*np.sqrt(252)*100
    sortino = ann_ret/sortino_d if sortino_d > 0 else 0
    calmar  = ann_ret/abs(max_dd) if max_dd != 0 else 0
    print(f"\n  {label}")
    print(f"    Annual return : {ann_ret:+.1f}%")
    print(f"    Annual vol    : {ann_vol:.1f}%")
    print(f"    Sharpe ratio  : {sharpe:.2f}")
    print(f"    Sortino ratio : {sortino:.2f}")
    print(f"    Calmar ratio  : {calmar:.2f}")
    print(f"    Max drawdown  : {max_dd:.1f}%")

print("\n── Performance Summary ─────────────────────────────────────")
compute_metrics(pd.Series(portfolio_values, index=portfolio_dates), "Regime-Shift Strategy")
compute_metrics(pd.Series(benchmark_values, index=portfolio_dates), "60/40 Benchmark")

# ── Plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
port_s = pd.Series(portfolio_values, index=portfolio_dates)
bm_s   = pd.Series(benchmark_values, index=portfolio_dates)

port_s.plot(ax=axes[0], color='#2ecc71', linewidth=1.5, label='Regime-Shift')
bm_s.plot(  ax=axes[0], color='#3498db', linewidth=1.5, label='60/40 Benchmark')
axes[0].set_title('Equity Curve: Regime-Shift vs 60/40 Benchmark', fontsize=13)
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].legend()
axes[0].grid(alpha=0.3)

dd_p = ((port_s - port_s.cummax()) / port_s.cummax()) * 100
dd_b = ((bm_s   - bm_s.cummax())   / bm_s.cummax())   * 100
dd_p.plot(ax=axes[1], color='#2ecc71', linewidth=1, label='Regime-Shift')
dd_b.plot(ax=axes[1], color='#3498db', linewidth=1, label='60/40')
axes[1].fill_between(port_s.index, dd_p, 0, color='#2ecc71', alpha=0.2)
axes[1].set_title('Drawdown (%)', fontsize=13)
axes[1].set_ylabel('Drawdown %')
axes[1].legend()
axes[1].grid(alpha=0.3)

reg_series = pd.Series([''] + regime_history, index=portfolio_dates)
cmap       = {'Bull':'#2ecc71','Bear':'#e67e22','Crisis':'#e74c3c',''  :'white'}
for date, regime in reg_series.items():
    if regime:
        axes[2].axvspan(date, date + pd.Timedelta(days=1),
                        color=cmap[regime], alpha=0.4, linewidth=0)
axes[2].set_title('Detected Regime During Backtest', fontsize=13)
axes[2].set_yticks([])
patches = [mpatches.Patch(color=cmap[r], label=r) for r in ['Bull','Bear','Crisis']]
axes[2].legend(handles=patches)
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../outputs/04_backtest.png', dpi=150)
plt.show()
print("\n✓ Phase 5 complete!  Saved → outputs/04_backtest.png")

In [ ]:
# ── PHASE 6: Professional Tearsheet ─────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
import pandas as pd
import numpy as np

results = pd.read_csv('../outputs/backtest_results.csv', index_col=0, parse_dates=True)
port    = results['portfolio']
bm      = results['benchmark']

def metrics(s):
    dr      = s.pct_change().dropna()
    ann_r   = ((s.iloc[-1]/s.iloc[0])**(252/len(dr))-1)*100
    ann_v   = dr.std()*np.sqrt(252)*100
    sharpe  = ann_r/ann_v
    dd      = (s - s.cummax())/s.cummax()*100
    maxdd   = dd.min()
    sortino = ann_r/(dr[dr<0].std()*np.sqrt(252)*100)
    calmar  = ann_r/abs(maxdd)
    wr      = (dr>0).mean()*100
    return dict(ann_r=ann_r, ann_v=ann_v, sharpe=sharpe,
                maxdd=maxdd, sortino=sortino, calmar=calmar, wr=wr)

mp = metrics(port)
mb = metrics(bm)

GREEN = '#a8ff3e'
BLUE  = '#00bfff'
RED   = '#ff4444'
GRAY  = '#444444'

fig = plt.figure(figsize=(18, 14), facecolor='#0d0d0d')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

title_kw = dict(color=GREEN, fontsize=11, fontweight='bold', pad=8)
label_kw = dict(color='#cccccc', fontsize=9)
tick_kw  = dict(colors='#888888', labelsize=8)

fig.text(0.5, 0.97, 'REGIME-SHIFT  ·  Macro-Aware Tactical Allocation Engine',
         ha='center', fontsize=16, fontweight='bold', color=GREEN, fontfamily='monospace')
fig.text(0.5, 0.945,
         f'Backtest: {port.index[0].date()} → {port.index[-1].date()}'
         '  |  Walk-forward  |  7 bps transaction cost  |  Monthly rebalance',
         ha='center', fontsize=9, color='#888888')

ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(port.index, port, color=GREEN, linewidth=1.2, label='Regime-Shift')
ax1.plot(bm.index,   bm,   color=BLUE,  linewidth=1.2, label='60/40 Benchmark', alpha=0.8)
ax1.set_facecolor('#111111')
ax1.set_title('Equity Curve', **title_kw)
ax1.set_ylabel('Portfolio Value ($)', **label_kw)
ax1.legend(fontsize=8, facecolor='#222222', labelcolor='white')
ax1.grid(alpha=0.15, color=GRAY)
ax1.tick_params(axis='both', **tick_kw)
for sp in ['top','right']: ax1.spines[sp].set_visible(False)
for sp in ['bottom','left']: ax1.spines[sp].set_color(GRAY)

ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor('#111111')
ax2.set_title('Performance Metrics', **title_kw)
ax2.axis('off')
rows = [
    ('Metric',        'Regime-Shift',         '60/40'),
    ('Annual Return', f"{mp['ann_r']:+.1f}%",  f"{mb['ann_r']:+.1f}%"),
    ('Annual Vol',    f"{mp['ann_v']:.1f}%",   f"{mb['ann_v']:.1f}%"),
    ('Sharpe',        f"{mp['sharpe']:.2f}",   f"{mb['sharpe']:.2f}"),
    ('Sortino',       f"{mp['sortino']:.2f}",  f"{mb['sortino']:.2f}"),
    ('Calmar',        f"{mp['calmar']:.2f}",   f"{mb['calmar']:.2f}"),
    ('Max Drawdown',  f"{mp['maxdd']:.1f}%",   f"{mb['maxdd']:.1f}%"),
    ('Win Rate',      f"{mp['wr']:.1f}%",      f"{mb['wr']:.1f}%"),
]
for i, (label, val_p, val_b) in enumerate(rows):
    y  = 0.95 - i * 0.11
    ih = (i == 0)
    ax2.text(0.02, y, label,  transform=ax2.transAxes,
             color=GREEN if ih else '#aaaaaa', fontsize=8.5,
             fontweight='bold' if ih else 'normal')
    ax2.text(0.55, y, val_p,  transform=ax2.transAxes,
             color=GREEN if ih else 'white',  fontsize=8.5,
             fontweight='bold' if ih else 'normal', ha='center')
    ax2.text(0.88, y, val_b,  transform=ax2.transAxes,
             color=BLUE  if ih else 'white',  fontsize=8.5,
             fontweight='bold' if ih else 'normal', ha='center')
    if not ih:
        ax2.add_line(mlines.Line2D([0,1],[y-0.04,y-0.04],
                     transform=ax2.transAxes, color=GRAY, linewidth=0.4))

ax3 = fig.add_subplot(gs[1, :2])
dd_p = (port - port.cummax()) / port.cummax() * 100
dd_b = (bm   - bm.cummax())   / bm.cummax()   * 100
ax3.fill_between(port.index, dd_p, 0, color=GREEN, alpha=0.3)
ax3.fill_between(bm.index,   dd_b, 0, color=BLUE,  alpha=0.2)
ax3.plot(port.index, dd_p, color=GREEN, linewidth=0.8)
ax3.plot(bm.index,   dd_b, color=BLUE,  linewidth=0.8)
ax3.set_facecolor('#111111')
ax3.set_title('Drawdown (%)', **title_kw)
ax3.set_ylabel('Drawdown %', **label_kw)
ax3.grid(alpha=0.15, color=GRAY)
ax3.tick_params(axis='both', **tick_kw)
for sp in ['top','right']: ax3.spines[sp].set_visible(False)
for sp in ['bottom','left']: ax3.spines[sp].set_color(GRAY)

ax4 = fig.add_subplot(gs[1, 2])
ax4.set_facecolor('#111111')
ax4.set_title('Monthly Returns (Strategy)', **title_kw)
monthly    = port.resample('ME').last().pct_change().dropna() * 100
monthly_df = monthly.to_frame('ret')
monthly_df['year']  = monthly_df.index.year
monthly_df['month'] = monthly_df.index.month
pivot = monthly_df.pivot_table(values='ret', index='year', columns='month')
pivot.columns = ['J','F','M','A','M','J','J','A','S','O','N','D']
ax4.imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=-5, vmax=5)
ax4.set_xticks(range(12))
ax4.set_xticklabels(pivot.columns, fontsize=7, color='#aaaaaa')
ax4.set_yticks(range(len(pivot.index)))
ax4.set_yticklabels(pivot.index, fontsize=7, color='#aaaaaa')
ax4.tick_params(length=0)
for sp in ax4.spines.values(): sp.set_visible(False)

ax5 = fig.add_subplot(gs[2, :2])
dr = port.pct_change().dropna()
rs = (dr.rolling(252).mean()*252) / (dr.rolling(252).std()*np.sqrt(252))
rs.plot(ax=ax5, color=GREEN, linewidth=1)
ax5.axhline(0, color=RED,  linewidth=0.8, linestyle='--')
ax5.axhline(1, color=GRAY, linewidth=0.8, linestyle='--', alpha=0.5)
ax5.fill_between(rs.index, rs, 0, where=rs>0, color=GREEN, alpha=0.2)
ax5.fill_between(rs.index, rs, 0, where=rs<0, color=RED,   alpha=0.2)
ax5.set_facecolor('#111111')
ax5.set_title('Rolling 1-Year Sharpe Ratio', **title_kw)
ax5.set_ylabel('Sharpe', **label_kw)
ax5.grid(alpha=0.15, color=GRAY)
ax5.tick_params(axis='both', **tick_kw)
for sp in ['top','right']: ax5.spines[sp].set_visible(False)
for sp in ['bottom','left']: ax5.spines[sp].set_color(GRAY)

ax6 = fig.add_subplot(gs[2, 2])
ax6.set_facecolor('#111111')
ax6.set_title('Regime Distribution', **title_kw)
reg    = results['regime'].replace('', np.nan).dropna()
counts = reg.value_counts()
cpie   = [{'Bull': GREEN, 'Bear':'#e67e22', 'Crisis': RED}.get(r,'gray')
          for r in counts.index]
_, texts, autotexts = ax6.pie(
    counts.values, labels=counts.index, colors=cpie,
    autopct='%1.0f%%',
    textprops=dict(color='white', fontsize=9),
    wedgeprops=dict(linewidth=0.5, edgecolor='#222222'))
for at in autotexts: at.set_color('#111111')

plt.savefig('../outputs/05_tearsheet.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

print("✓ Tearsheet saved → outputs/05_tearsheet.png")
print("\n🎉 REGIME-SHIFT PROJECT COMPLETE!")
print("   All outputs saved in the outputs/ folder.")